# 조리로봇 화재 인식 — 게이트 테스트 재현

실제 급식실 프레임 + 실제 화염 영상을 합성해 학습하고, **공개 실화재 데이터셋으로 검증**합니다.
학습에 실제 화재 이미지는 한 장도 쓰지 않습니다.

**런타임 → 런타임 유형 변경 → GPU(T4)** 로 설정한 뒤 위에서부터 실행하세요. 전체 약 1시간.

## 1. 저장소와 소재 준비

In [ ]:
!pip -q install ultralytics==8.3.* kagglehub

import os, glob
# 저장소 압축본(kitchen-fire-poc.zip)과 소재(assets_1~3.zip)를 업로드합니다.
from google.colab import files
for z in files.upload():
    !unzip -q -o "{z}" -d /content/poc
%cd /content/poc
print(sorted(os.listdir('.')))
print('bases', len(glob.glob('assets/bases/*.jpg')),
      '| flamelib', len(glob.glob('assets/flamelib/*.webp')),
      '| negsrc', len(glob.glob('assets/negsrc/*.jpg')),
      '| eval_neg', len(glob.glob('assets/eval_neg/*.jpg')))

## 2. D-Fire 내려받기

Kaggle 계정이 필요합니다. kaggle.com → Settings → API → **Create Legacy API Key** 로
`kaggle.json` 을 받아 아래 셀에서 선택하세요.

In [ ]:
import json, os
from google.colab import files
files.upload()                     # kaggle.json 선택
cfg = json.load(open('kaggle.json'))
os.environ['KAGGLE_USERNAME'], os.environ['KAGGLE_KEY'] = cfg['username'], cfg['key']

import kagglehub
DFIRE = kagglehub.dataset_download('sayedgamal99/smoke-fire-detection-yolo')
print('D-Fire:', DFIRE)

## 3. 평가셋 구성

선별 규칙은 성능을 보기 전에 확정한 것으로, 이후 수정하지 않습니다.
평가에 쓰는 이미지는 학습에 절대 들어가지 않습니다(`train_bg.txt` 가 그 분리를 담당).

In [ ]:
!python scripts/dfire_eval_set.py --dfire "{DFIRE}" --out eval

## 4. 합성 학습셋 생성

주방 베이스 70장에 화염 390종을 합성하고, 주방 밖 배경 600장에도 같은 화염을 얹습니다.
바운딩박스는 합성 좌표에서 자동 생성되므로 수작업 라벨링이 없습니다.

In [ ]:
!python scripts/synthesize.py --assets assets --out ds \
    --dfire-bg-list eval/train_bg.txt --dfire-bg-count 600

## 5. 학습 (T4 기준 약 50분)

In [ ]:
!yolo detect train model=yolov8s.pt data=/content/poc/ds/data.yaml \
    epochs=60 imgsz=640 batch=16 seed=0 val=False project=/content/runs name=gate

## 6. 채점

세 그룹을 나눠 재는 이유는 실패 원인을 구분하기 위해서입니다.
A가 낮고 C도 낮으면 전이 실패, A가 낮은데 C가 높으면 도메인 암기, B가 높으면 조리 중 오작동입니다.

In [ ]:
import glob, os
best = max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime)
print('모델:', best)
!python scripts/eval_gate.py --weights "{best}" --eval-dir eval --cctv assets/eval_neg

## 7. 오탐 위치 확인

숫자만 보지 말고 무엇을 화재로 오인하는지 눈으로 봅니다.
1회차에서는 기름이 아니라 **장비의 주황색 손잡이와 제어판 표시등**을 잡고 있었고,
그 관찰이 negative 다양성이라는 해법으로 이어졌습니다.

In [ ]:
import cv2, numpy as np, glob
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

m = YOLO(best); CONF = 0.25
hits = [p for p in sorted(glob.glob('assets/eval_neg/*.jpg'))
        if len(m.predict(p, conf=CONF, verbose=False)[0].boxes)][:6]
print(f'오탐 {len(hits)}장 표시')
t = []
for p in hits:
    im = cv2.imread(p)
    for b in m.predict(p, conf=CONF, verbose=False)[0].boxes.xyxy.cpu().numpy().astype(int):
        cv2.rectangle(im, (b[0], b[1]), (b[2], b[3]), (0, 0, 255), 3)
    t.append(cv2.resize(im, (400, 300)))
while len(t) % 3: t.append(np.zeros((300, 400, 3), np.uint8))
if t: cv2_imshow(np.vstack([np.hstack(t[i:i+3]) for i in range(0, len(t), 3)]))